In [ ]:
import numpy as np
import torch
from torch import Tensor
from typing import Optional,Any,Union,Callable
import torch.nn as nn
import torch.nn.functional as F
import math,copy,time

class MultiHeadAttention(nn.Module):
    def __init__(self,
                num_heads:int,
                d_model:int,
                dropout:float=0.1):
        super(MultiHeadAttention,self).__init__()
        assert d_model % num_heads == 0 ,"d_model must be divisible by num_heads "
        self.k_dim = d_model // num_heads
        self.num_heads = num_heads
        self.proj_weights = clones(nn.Liear(d_model,d_model),4)
        self.attention_score = None
        self.dropout = nn.Dropout(p=dropout)
        
    def forward(self,query:Tensor,key:Tensor,value:Tensor,mask:Optional[Tensor]=None):
        """ 
        Args: 
            query: shape (batch_size, seq_len, d_model) 
            key: shape (batch_size, seq_len, d_model) 
            value: shape (batch_size, seq_len, d_model) 
            mask: shape (batch_size, seq_len, seq_len). Since we assume all data use a same mask, so here the shape also equals to (1, seq_len, seq_len) 
        Return: 
            out: shape (batch_size, seq_len, d_model). The output of a multihead attention layer 
         """
        
        if mask is not None: # (batch_size, seq_len, seq_len)
            mask = mask.unsqueeze(1)  # 增加维度，匹配scores的形状 -> [batch, 1, seq, seq]
        batch_size = query.size(0)
        # 1) Apply W^Q, W^K, W^V to generate new query, key, value
        query, key, value = [proj_weight(x).view(batch_size, -1, self.num_heads, self.k_dim).transpose(1, 2) 
                 for proj_weight, x in zip(self.proj_weights, [query, key, value])] # -1 equals to seq_len
        """
        # 等价展开（易理解）
            query = self.proj_weights[0](query).view(batch_size, -1, self.num_heads, self.k_dim).transpose(1, 2)
            key   = self.proj_weights[1](key)  .view(batch_size, -1, self.num_heads, self.k_dim).transpose(1, 2)
            value = self.proj_weights[2](value).view(batch_size, -1, self.num_heads, self.k_dim).transpose(1, 2)
        
        """
        # 2）计算注意力分数和输出
        out,self.attention_score = attention(query,key,value,mask=mask,dropout=self.dropout)
        
        # 3) "concat" 输出
        out = out.transpose(1,2).contiguous()\
            .view(batch_size,-1,self.num_head * self.k_dim)
        # 4) apply W^O得到最终的output
        out = self.proj_weights[-1](out)
        return out
    
    def clones(module,N):
        # 生成N个相同的层
        return nn.ModuleList([copy.deepcopy(model) for _ in range(N)])
    
    def attention(query:Tensor,
                 key:Tensor,
                 value:Tensor,
                 mask:Optional[Tensor]= None,
                 dropout:float = 0.1):
        """
        定义如何计算注意力分数
        args:
            query: shape (batch_size, num_heads, seq_len, k_dim) 
            key: shape(batch_size, num_heads, seq_len, k_dim) 
            value: shape(batch_size, num_heads, seq_len, v_dim) 
            mask: shape (batch_size, num_heads, seq_len, seq_len). Since our assumption, here the shape is (1, 1, seq_len, seq_len)
            
        Return:
            out:shape (batch_size,v_dim) 输出注意力的头
            attention_score : shape （seq_len,seq_len）
        """
        k_dim  =query.size(-1)
        
        # shape (seq_len,seq_len) , row:token col: 代表这个token的注意力分数
        scores = torch.matmul(query,key.transpose(-2,-1)) / math.sqrt(k_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0 ,-1e10) # 将mask掉的替换为非常小的值
        attention_score = F.softmax(scores,dim = -1)
        
        if dropout is not None:
            attention_score = dropout(attention_score)
        out = torch.matmul(attention_score,value)
        return out,attention_score # shape :(seq_len,v_dim),(seq_len,seq_len)
    
# if __name__ == '__main__':

d_model = 8
seq_len = 3
batch_size = 6
num_heads = 2

# mask = None
# 创建上三角矩阵（全1），然后取反得到下三角
mask = torch.tril(torch.ones((seq_len,seq_len)),diagonal = 0).unsqueeze(0)

input = torch.rand(batch_size,seq_len,d_model)
multi_attn = MultiHeadAttention(num_heads = num_heads,d_model = d_model,dropout = 0.1)
out = multi_attn(query = input,key=input,value=input,mask=mask)
print(out.shape)